# EDA: Alfabank credit score

## Описание файлов и полей датасета

### 1. `{train/test}_target.csv`
Тренировочная и тестовая выборка кредитов.

* `id` — идентификатор заявки на кредит
* `flag` — целевая переменная: `1` — факт ухода клиента в дефолт (присутствует только в файле `train_target.csv`)

### 2. `{train/test}_data`
Данные кредитных историй для тренировки и обучения модели.

**Идентификаторы:**
* `id` — идентификатор заявки
* `rn` — порядковый номер кредитного продукта в кредитной истории

**Временные интервалы (в днях):**
* `pre_since_opened` — дней с даты открытия кредита до даты сбора данных
* `pre_since_confirmed` — дней с даты подтверждения информации по кредиту до даты сбора данных
* `pre_pterm` — плановое количество дней с даты открытия кредита до даты закрытия
* `pre_fterm` — фактическое количество дней с даты открытия кредита до даты закрытия
* `pre_till_pclose` — плановое количество дней с даты сбора данных до даты закрытия
* `pre_till_fclose` — фактическое количество дней с даты сбора данных до даты закрытия

**Суммы и лимиты:**
* `pre_loans_credit_limit` — кредитный лимит
* `pre_loans_next_pay_summ` — сумма следующего платежа по кредиту
* `pre_loans_outstanding` — оставшаяся невыплаченная сумма кредита
* `pre_loans_total_overdue` — текущая просроченная задолженность
* `pre_loans_max_overdue_sum` — максимальная просроченная задолженность
* `pre_loans_credit_cost_rate` — полная стоимость кредита

**Просрочки и флаги отсутствия просрочек:**
* `pre_loans5` / `is_zero_loans_5` — число просрочек до 5 дней / флаг их отсутствия
* `pre_loans530` / `is_zero_loans_530` — число просрочек от 5 до 30 дней / флаг их отсутствия
* `pre_loans3060` / `is_zero_loans_3060` — число просрочек от 30 до 60 дней / флаг их отсутствия
* `pre_loans6090` / `is_zero_loans_6090` — число просрочек от 60 до 90 дней / флаг их отсутствия
* `pre_loans90` / `is_zero_loans90` — число просрочек более 90 дней / флаг их отсутствия

**Отношения (доли):**
* `pre_util` — отношение оставшейся невыплаченной суммы кредита к кредитному лимиту
* `pre_over2limit` — отношение текущей просроченной задолженности к кредитному лимиту
* `pre_maxover2limit` — отношение максимальной просроченной задолженности к кредитному лимиту
* `is_zero_util` — флаг: отношение оставшейся суммы к лимиту равно 0
* `is_zero_over2limit` — флаг: отношение текущей просрочки к лимиту равно 0
* `is_zero_maxover2limit` — флаг: отношение максимальной просрочки к лимиту равно 0

**Категориальные и закодированные признаки:**
* `enc_paym_{0..n}` — статусы ежемесячных платежей за последние `n` месяцев
* `enc_loans_account_holder_type` — тип отношения к кредиту
* `enc_loans_credit_status` — статус кредита
* `enc_loans_account_cur` — валюта кредита
* `enc_loans_credit_type` — тип кредита

**Дополнительные флаги:**
* `pclose_flag` — флаг: плановое количество дней с даты открытия до даты закрытия не определено
* `fclose_flag` — флаг: фактическое количество дней с даты открытия до даты закрытия не определено

### 3. `description.xlsx`
Описание полей в `{train/test}_target.csv` и `{train/test}_data`.

### 4. `sample_submission.csv`
Пример сабмита для отправки на соревнование.

* `id` — идентификатор заявки на кредит
* `score` — предсказание модели  

***

**Ссылка на датасет: https://ods.ai/competitions/dl-fintech-bki/data**

***

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport

In [2]:
files = [f'./data_for_competition/train_data/train_data_{i}.pq' for i in range(12)]

tx = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True,
)

target = pd.read_csv('./data_for_competition/train_target.csv')

In [3]:
target.head(3)

,id,flag
0,0,0
1,1,0
2,2,0


In [4]:
tx.head(15)

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,...,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,...,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,...,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12,2,...,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10,2,...,3,3,3,4,1,3,4,1,0,0
5,0,6,5,0,11,8,12,11,4,2,...,3,3,3,4,1,2,3,1,0,1
6,0,7,3,9,1,2,12,14,15,5,...,3,3,3,4,1,3,4,1,0,0
7,0,8,2,9,2,3,12,14,15,5,...,3,3,3,4,1,3,4,1,0,0
8,0,9,1,9,11,13,14,8,2,5,...,3,3,3,4,1,2,4,1,0,0
9,0,10,7,9,2,10,8,8,16,4,...,3,3,3,4,1,2,4,1,0,0


In [5]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26162717 entries, 0 to 26162716
Data columns (total 61 columns):
 #   Column                         Dtype
---  ------                         -----
 0   id                             int64
 1   rn                             int64
 2   pre_since_opened               int64
 3   pre_since_confirmed            int64
 4   pre_pterm                      int64
 5   pre_fterm                      int64
 6   pre_till_pclose                int64
 7   pre_till_fclose                int64
 8   pre_loans_credit_limit         int64
 9   pre_loans_next_pay_summ        int64
 10  pre_loans_outstanding          int64
 11  pre_loans_total_overdue        int64
 12  pre_loans_max_overdue_sum      int64
 13  pre_loans_credit_cost_rate     int64
 14  pre_loans5                     int64
 15  pre_loans530                   int64
 16  pre_loans3060                  int64
 17  pre_loans6090                  int64
 18  pre_loans90                    int64
 19

In [6]:
tx.isna().sum().sum()

np.int64(0)

In [7]:
tx.duplicated().sum()

np.int64(0)

In [8]:
target.nunique()

id      3000000
flag          2
dtype: int64

In [9]:
tx['id'].nunique()

3000000

In [10]:
for col in tx.columns:
    tx[col] = pd.to_numeric(tx[col], downcast='integer')

In [11]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26162717 entries, 0 to 26162716
Data columns (total 61 columns):
 #   Column                         Dtype
---  ------                         -----
 0   id                             int32
 1   rn                             int8 
 2   pre_since_opened               int8 
 3   pre_since_confirmed            int8 
 4   pre_pterm                      int8 
 5   pre_fterm                      int8 
 6   pre_till_pclose                int8 
 7   pre_till_fclose                int8 
 8   pre_loans_credit_limit         int8 
 9   pre_loans_next_pay_summ        int8 
 10  pre_loans_outstanding          int8 
 11  pre_loans_total_overdue        int8 
 12  pre_loans_max_overdue_sum      int8 
 13  pre_loans_credit_cost_rate     int8 
 14  pre_loans5                     int8 
 15  pre_loans530                   int8 
 16  pre_loans3060                  int8 
 17  pre_loans6090                  int8 
 18  pre_loans90                    int8 
 19

In [12]:
with pd.option_context('display.max_rows', None):
    display(tx.describe(include='all').round(2).T)

,count,mean,std,min,25%,50%,75%,max
id,26162717.0,1545556.97,867988.04,0.0,794902.0,1568512.0,2303681.0,2999999.0
rn,26162717.0,7.05,5.40,1.0,3.0,6.0,10.0,58.0
pre_since_opened,26162717.0,9.36,5.78,0.0,4.0,9.0,14.0,19.0
pre_since_confirmed,26162717.0,8.22,4.71,0.0,4.0,9.0,12.0,17.0
pre_pterm,26162717.0,8.37,5.32,0.0,4.0,8.0,14.0,17.0
pre_fterm,26162717.0,8.33,4.48,0.0,6.0,8.0,12.0,16.0
pre_till_pclose,26162717.0,7.14,5.15,0.0,2.0,7.0,12.0,16.0
pre_till_fclose,26162717.0,8.36,4.22,0.0,5.0,10.0,11.0,15.0
pre_loans_credit_limit,26162717.0,9.68,5.83,0.0,5.0,10.0,15.0,19.0
pre_loans_next_pay_summ,26162717.0,2.36,1.31,0.0,2.0,2.0,2.0,6.0


In [13]:
target['flag'].value_counts(normalize=True) # дисбаланс

flag
0    0.964519
1    0.035481
Name: proportion, dtype: float64

In [ ]:
# Пространство для Aggregation-based Feature Engineering

agg_dict = { 
    
}

# agg_df = tx.groupby('id').agg(agg_dict)
# agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
# agg_df = agg_df.reset_index()

***

### Стратегии работы с датасетом

Данный датасет представляет собой временную последовательность поведения клиента при взятии и выплате кредитов, в связи с чем существует две стратегии работы с данными:

- **Агрегация данных (переход к табличному формату):** 
  У нас 26 миллионов строк, но всего 3 миллиона уникальных клиентов. Мы можем агрегировать имеющиеся данные, предварительно их проанализировав и выделив новые признаки, для построения классических моделей на табличных данных.
  
- **Использование последовательностей (Sequence Models):** 
  Либо мы можем использовать таблицу как есть для построения нейросетевых моделей, например, RNN, сохраняя хронологию и порядок событий.  

#### Краткие выводы

- Пропусков, дубликатов и аномальных значений не обнаружено, проведен `downcasting` типов для экономии памяти.  
- Сильный дисбаланс целевой переменной (`0.965` против `0.035`).
- Совместное распределение признаков и матрицу корреляций имеет смысл смотреть после агрегации, в случае использования RNN потребуеются иные подходы к EDA.

***